In [ ]:
%reload_ext autoreload
%autoreload 2

import sys
import os

# Adjust the path to your project root if needed
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import weibull_min
from functions.functions_dp import *
import opendp.prelude as dp
import pickle
import os
dp.enable_features("contrib")

merged_df = pd.read_csv('../data/merged_data.csv')

In [ ]:
epsilon = 3.84
gas_consumption = merged_df['2020']
merged_df['gas_consumption'] = merged_df['2020']
sensitivity = gas_consumption.max() - gas_consumption.min()
print("sensitivity:", sensitivity)
Q1 = np.percentile(gas_consumption, 25)
Q3 = np.percentile(gas_consumption, 75)
IQR = Q3 - Q1
upper_bound = Q3 + 1.5 * IQR
print("upper bound:", upper_bound)
bounds = (0., upper_bound)
base_lap = dp.m.make_laplace(
    dp.atom_domain(T=float),
    dp.absolute_distance(T=float),
    scale=sensitivity/epsilon
)

absolute_distance = sensitivity
print("epsilon:", base_lap.map(d_in=absolute_distance))

input_domain = dp.vector_domain(dp.atom_domain(bounds=bounds))
bounded_sum = dp.t.make_sum(input_domain, dp.symmetric_distance())

max_contributions = 1
print(bounded_sum.map(d_in=max_contributions))
bounded_sum.map(d_in=max_contributions)

input_domain = dp.vector_domain(dp.atom_domain(T=float))
input_metric = dp.symmetric_distance()

# call the constructor to produce the transformation `clamp`
clamp = dp.t.make_clamp(input_domain, input_metric, bounds=bounds)

In [ ]:
dp_sum = clamp >> bounded_sum >> base_lap

# compute the DP sum of a dataset of bounded elements
print("DP sum:", max(0, dp_sum(gas_consumption[0:40].to_list())))

# evaluate the privacy loss of the dp_sum, when an individual can contribute at most 2 records
print("epsilon:", dp_sum.map(d_in=1))

In [ ]:
# list_dfs = []
# list_dfs_noisy = []
# n_groups = [1, 2, 4, 5, 10, 20, 25, 50, 100]
# merged_df['gas_consumption'] = merged_df['2020']
# for i in n_groups:
#     df, noisy_df = calculate_grouped_mean_gas_consumption(merged_df, n_groups=i, chain=dp_sum)
#     list_dfs.append(df)
#     list_dfs_noisy.append(noisy_df)

In [ ]:
# heat_density_limit = 2e5
# df_high_density_list = []
# noisy_df_high_density_list = []
# plot_choice = 'gas_consumption' # 'heat_density' or 'gas_consumption'
# for i in range(len(n_groups)):
#     df_high_density = list_dfs[i][list_dfs[i]['gas_consumption'] > heat_density_limit]
#     df_high_density_list.append(df_high_density)
#     noisy_df_high_density = list_dfs_noisy[i][list_dfs_noisy[i]['gas_consumption'] > heat_density_limit]
#     noisy_df_high_density_list.append(noisy_df_high_density)
#     plot_limit(list_dfs[i], list_dfs_noisy[i], df_high_density, noisy_df_high_density, i, f'images/real/group_size_{n_groups[i]}.png', plot_choice)

In [ ]:
group_sizes = [1, 2, 5, 10, 20, 25, 50, 100]
epsilon_values = np.logspace(-3, 3, num=500)
heat_density_limit = 2e5
# Function to evaluate and save results
def evaluate_and_save(df_generated, group_sizes, epsilon_values, heat_density_limit, sensitivity, bounded_sum, clamp, iterations, save_interval=2, filename='../data/all_results_real.pickle'):
    # Load existing results if the file exists
    if os.path.exists(filename):
        with open(filename, 'rb') as handle:
            all_results = pickle.load(handle)
            print(all_results)
        # Determine the last group size processed
        if not all_results.empty:
            last_group_size_done = all_results['n_groups'].max()
            start_index = list(group_sizes).index(last_group_size_done) + 1
        else:
            start_index = 0
    else:
        all_results = pd.DataFrame()
        start_index = 0
    
    # Continue evaluation from the next group size
    for i, group_size in enumerate(group_sizes[start_index:], start=start_index):
        # Evaluate for the current group size
        results = evaluate_over_epsilon(df_generated, [group_size], epsilon_values, heat_density_limit, sensitivity, bounded_sum, clamp, iterations, calculate_grouped_mean_gas_consumption)
        # Append the results to the all_results DataFrame
        all_results = pd.concat([all_results, pd.DataFrame(results)], ignore_index=True)
        
        # Save intermediate results every save_interval iterations
        if (i + 1) % save_interval == 0 or (i + 1) == len(group_sizes):
            with open(filename, 'wb') as handle:
                pickle.dump(all_results, handle, protocol=pickle.HIGHEST_PROTOCOL)
    return all_results

# Evaluate and save results
all_results = evaluate_and_save(merged_df, group_sizes, epsilon_values, heat_density_limit, sensitivity, bounded_sum, clamp, 100)

plot_epsilon(all_results)
# plot_rmse_mape(all_results)

In [ ]:
plt.figure(figsize=(14, 6))
for n_groups in [1, 5, 10, 20, 25, 50, 100]:
    results = all_results[all_results['n_groups'] == n_groups]
    epsilon_values = np.array(results['epsilon'])
    rmse_mean = np.array(results['rmse_mean'])
    rmse_std = np.array(results['rmse_std'])
    rmse_perc_mean = np.array(results['rmse_perc_mean'])
    rmse_perc_std = np.array(results['rmse_perc_std'])
    mape_mean = np.array(results['mape_mean'])
    mape_std = np.array(results['mape_std'])
    accuracy_mean = np.array(results['accuracy_mean'])
    accuracy_std = np.array(results['accuracy_std'])

    # Plot RMSE
    plt.subplot(1, 2, 2)
    plt.xlim(10**-2, 1000)
    plt.ylim(0, 100)
    plt.plot(epsilon_values, rmse_perc_mean, label=f'Group size = {int(100/n_groups)}')
    plt.xscale('log')
    plt.xlabel('Epsilon', fontsize=15)
    plt.ylabel('RMSE / Mean (%)', fontsize=15)
    plt.title('RMSE / Mean per Epsilon', fontsize=16)
    formatter = ticker.ScalarFormatter(useMathText=True)
    formatter.set_powerlimits((0, 0))
    plt.gca().yaxis.set_major_formatter(formatter)
    plt.xticks(fontsize=15)  # Set x-axis tick label size
    plt.yticks(fontsize=15)  # Set y-axis tick label size
    plt.grid(True)
    plt.legend(fontsize=15)

    # Plot Accuracy
    plt.subplot(1, 2, 1)
    plt.plot(epsilon_values, accuracy_mean, label=f'Group size = {int(100/n_groups)}')
    plt.ylim(0, 105)
    plt.xlim(10**-2, 1000)
    plt.xscale('log')
    plt.xlabel('Epsilon', fontsize=15)
    plt.ylabel('Accuracy (%)', fontsize=15)
    plt.title('Accuracy per Epsilon', fontsize=16)
    plt.xticks(fontsize=15)  # Set x-axis tick label size
    plt.yticks(fontsize=15)  # Set y-axis tick label size
    plt.grid(True)
    plt.legend(fontsize=15)

plt.tight_layout()
plt.show()

In [ ]:
last_values = []
for n_groups in sorted(all_results['n_groups'].unique()):
    results = all_results[all_results['n_groups'] == n_groups]
    last_rmse_perc = results['rmse_perc_mean'].iloc[-1]
    last_values.append((n_groups, last_rmse_perc))

# Creating the LaTeX table
latex_table = r"""
\begin{table}[h!]
    \centering
    \begin{tabular}{|c|c|c|}
        \hline
        Group Size & Final RMSE/Mean (\%) \\
        \hline
"""

# Add rows to the LaTeX table
for group_size, rmse in last_values:
    latex_table += f"        {100/group_size} & {rmse} \\\\\n"

# Finalize the LaTeX table
latex_table += r"""        \hline
    \end{tabular}
    \caption{Final Accuracy and RMSE/Mean values for the last $\epsilon$ of each group size}
    \label{tab:final_accuracy_rmse}
\end{table}
"""

# Print the LaTeX table string
print(latex_table)